# R5.4 — 黑潮路径表示真实性与 R5.3 误差归因审计（不重训）

**研究问题**：在 R5.1–R5.3 的空间卷积修改未能稳定超过同参数量 LSTM 后，优先判断是否存在双分支路径重建的多值性、外推，以及异常路径与预测误差的关联。保留所有原始样本，不按测试误差自动剔除数据。

这个 Notebook 会生成：逐日轨迹经纬线交点数量 QC、逐年异常分布、双分支插值/物理范围诊断、R5.3 按年份/提前期/异常状态的误差分层、已训练模型的**成对**表现分析、中英双语论文风格图、含 SHA256 的自动 ZIP 归档。默认检查已冻结轨迹的代表性子集；可将 `TRACK_SCAN_LIMIT=0` 改为全样本扫描。

**重要**：多次穿越一条经纬线可能是合法的黑潮大蛇行，检测到多值性不等于轨迹错误。宽松经纬范围异常同样不能直接判为标注错误。此 Notebook 是诊断性的，不生成新的训练成绩，不冒充独立真值验证。

In [ ]:
# ===== 0. WSL 工作目录 / 输入输出 =====
from pathlib import Path
from dataclasses import dataclass, asdict
import json, io, zipfile, hashlib, warnings, gc, sys
from IPython.display import display
import numpy as np
import pandas as pd

@dataclass
class CFG:
    ROOT: str = '/mnt/d/KuroshioDaily1993_2025'
    DATASET_REL: str = 'final/dataset/daily_dual_branch_paths_1993_2025_FINAL.npz'
    TRACK_DIR_REL: str = 'final/tracks_final'
    R53_DATA_REL: str = 'R5_3_空间卷积双通道资格实验/实验数据'
    R53_ZIP: str = 'R53_上传给ChatGPT_机制资格实验结果.zip'
    OUT_REL: str = 'R5_4_路径表示与预测误差归因审计'
    TRACK_SCAN_LIMIT: int = 1200  # 0为扫描全部; 各年份等比例抽取，并优先涵盖2008/2016异常日
    BOOTSTRAP_REPS: int = 1000
    BOOTSTRAP_BLOCK_DAYS: int = 30
    SEED: int = 5401
    FIG_DPI: int = 300
    MAX_ARCHIVE_FILE_MB: int = 150
cfg=CFG()
ROOT=Path(cfg.ROOT);OUT=ROOT/cfg.OUT_REL;DATA=OUT/'实验数据';REPORT=OUT/'报告';FZH=OUT/'图表'/'中文';FEN=OUT/'图表'/'英文'
for p in [DATA,REPORT,FZH,FEN]:p.mkdir(parents=True,exist_ok=True)
DS=ROOT/cfg.DATASET_REL;TRACKS=ROOT/cfg.TRACK_DIR_REL
RD=ROOT/cfg.R53_DATA_REL;RZIP=ROOT/cfg.R53_ZIP
print('Python:',sys.executable,'ROOT:',ROOT,'Frozen dataset:',DS.exists(),'Tracks:',TRACKS.exists())
print('R53 local CSV:',RD.exists(),'R53 upload ZIP:',RZIP.exists())
if not (RD.exists() or RZIP.exists()):
    raise FileNotFoundError('找不到 R5.3 实验数据文件夹和 R53_上传给ChatGPT_机制资格实验结果.zip；请把R53 ZIP放到项目根目录。')


In [ ]:
# ===== 1. R5.3 真实结果与异常日期 =====
import matplotlib
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.font_manager import FontProperties

def load_r53_csv(stem):
    file=RD/stem
    if file.exists():return pd.read_csv(file,encoding='utf-8-sig')
    with zipfile.ZipFile(RZIP) as z:
        matches=[name for name in z.namelist() if name.endswith('/'+stem)]
        if len(matches)!=1:raise FileNotFoundError(f'R53 zip中未唯一找到 {stem}')
        return pd.read_csv(io.BytesIO(z.read(matches[0])),encoding='utf-8-sig')

qa=load_r53_csv('R53_03_机制资格QA逐样本误差.csv')
fl=load_r53_csv('R53_05_机制资格硬门槛.csv')
raw_flags=load_r53_csv('01_待人工复查的物理范围异常路径.csv')
coverage=load_r53_csv('01_逐年覆盖率.csv')
qa['起报日期']=pd.to_datetime(qa['起报日期']);qa['目标日期']=pd.to_datetime(qa['目标日期']);raw_flags['异常日期']=pd.to_datetime(raw_flags['异常日期'])
flag_set=set(raw_flags['异常日期'])
qa['异常目标日期']=qa['目标日期'].isin(flag_set)
raw_flags['年份']=raw_flags['异常日期'].dt.year
coverage=coverage.rename(columns={'年份':'年份'})
annual_flags=coverage.merge(raw_flags.groupby('年份').size().rename('标记日期数'),on='年份',how='left').fillna({'标记日期数':0})
annual_flags['标记比例']=annual_flags['标记日期数']/annual_flags['有效天数']
annual_flags.to_csv(DATA/'01_逐年异常路径比例.csv',index=False,encoding='utf-8-sig')
qa['单样本RMSE_km']=np.sqrt(qa['横向路径误差MSE_km2'].clip(lower=0))
fl.to_csv(DATA/'00_R53_资格判定原始表.csv',index=False,encoding='utf-8-sig')
print('R53 QA rows',len(qa),'flagged total',len(raw_flags),'QA years',sorted(qa['QA年份'].unique()))
display(fl);display(annual_flags)


In [ ]:
# ===== 2. 按年份 / 提前期 / 异常标记分层；不能把异常标记等同于错误 =====
# 按原R53实际使用的MSE先求平均再开方；不同seed与年份显式保留
strata=qa.groupby(['QA年份','模型','提前期_天','异常目标日期','种子'],as_index=False).agg(
    MSE均值=('横向路径误差MSE_km2','mean'),样本数=('横向路径误差MSE_km2','size'))
strata['RMSE_km']=np.sqrt(strata['MSE均值'])
strata.to_csv(DATA/'02_年份提前期异常状态分层误差_逐种子.csv',index=False,encoding='utf-8-sig')
strata_avg=strata.groupby(['QA年份','模型','提前期_天','异常目标日期'],as_index=False).agg(
    RMSE均值_km=('RMSE_km','mean'),随机种子数=('种子','size'),最少样本数=('样本数','min'))
strata_avg.to_csv(DATA/'02_年份提前期异常状态分层误差_汇总.csv',index=False,encoding='utf-8-sig')

# 原实验每个fold里相同issue、seed、lead的成对差异。只做描述性对比，不把2个年份冒充独立大样本。
selected=['Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv','R53-ChannelConv-PCConcat']
q=qa[qa['模型'].isin(selected)]
pairs=q.pivot_table(index=['QA年份','种子','起报日期','目标日期','提前期_天','异常目标日期'],columns='模型',values='横向路径误差MSE_km2',aggfunc='mean').reset_index()
for m in ['Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM']:
    pairs[f'PCConcat减{m}_MSE_km2']=pairs['R53-ChannelConv-PCConcat']-pairs[m]
pairs.to_csv(DATA/'03_同一事件配对模型误差差.csv',index=False,encoding='utf-8-sig')
# 直接在相同QA样本上比较：负值代表PCConcat更好
summary=pairs.groupby(['QA年份','提前期_天','异常目标日期'],as_index=False).agg(
    PCConcat相对MatchedLSTM_MSE差=('PCConcat减Matched-LSTM_MSE_km2','mean'),
    PCConcat相对原ETL_MSE差=('PCConcat减EOF-TCN-LSTM_MSE_km2','mean'),
    事件记录数=('PCConcat减Matched-LSTM_MSE_km2','size'))
summary.to_csv(DATA/'03_配对误差差分层汇总.csv',index=False,encoding='utf-8-sig')
display(strata_avg.head(20));display(summary)


In [ ]:
# ===== 3. 原始冻结轨迹：固定经纬线穿越次数、外推风险（诊断性） =====
# 此处仅检测某轴曲线与各固定经/纬线的交点数：0意味着当前曲线未覆盖该取值，>=2意味着存在多值分支。
# 多值可由真实蛇行导致，不能单靠多值就修改轨迹或删样本。
track_diagnostics=pd.DataFrame()
if not DS.exists() or not TRACKS.exists():
    warnings.warn('冻结NPZ或tracks_final目录不存在：跳过几何审计，但R53误差分层和图表仍可完成。')
else:
    with np.load(DS,allow_pickle=False) as z:
        dates=pd.to_datetime(z['dates']); lat_grid=np.asarray(z['branch1_target_lats'],float)[12:]
        lon_grid=np.asarray(z['branch2_target_lons'],float)
        obs1=np.asarray(z['branch1_lon'],float)[:,12:]
        obs2=np.asarray(z['branch2_lat'],float)
    keep2=lon_grid<=145.+1e-8;lon_grid=lon_grid[keep2];obs2=obs2[:,keep2]
    obs_dates={pd.Timestamp(d).normalize():i for i,d in enumerate(dates)}
    paths=sorted(TRACKS.glob('*.npz'))
    if not paths:raise RuntimeError('tracks_final目录下无NPZ，请核查是否冻结成功。')
    if cfg.TRACK_SCAN_LIMIT and len(paths)>cfg.TRACK_SCAN_LIMIT:
        sampled=[]
        for yy in range(1993,2026):
            pyy=[p for p in paths if p.stem[:4]==str(yy)]
            if pyy:
                n=max(1,round(cfg.TRACK_SCAN_LIMIT*len(pyy)/len(paths)))
                sampled += [pyy[int(i)] for i in np.linspace(0,len(pyy)-1,min(n,len(pyy)),dtype=int)]
        # QA年份中额外强制包括全部“物理异常”日期（如果轨迹存在），避免漏掉高风险样本
        focus=raw_flags[raw_flags['年份'].isin([2008,2016])]['异常日期']
        sampled += [TRACKS/f'{dt:%Y%m%d}.npz' for dt in focus if (TRACKS/f'{dt:%Y%m%d}.npz').exists()]
        paths=sorted(set(sampled))
    print('geometry scan tracks:',len(paths),'of',len(list(TRACKS.glob('*.npz'))))
    def count_crossings(vals,grid):
        a=np.asarray(vals[:-1],float);b=np.asarray(vals[1:],float);grid=np.asarray(grid,float)
        good=np.isfinite(a)&np.isfinite(b)&(np.abs(a-b)>1e-10)
        a=a[good,None];b=b[good,None]
        if len(a)==0:return np.zeros(len(grid),np.int16)
        g=grid[None,:]
        crossings=((a<=g)&(b>g))|((b<=g)&(a>g))
        return crossings.sum(axis=0)
    def unique_crossing_disagreement(coord,other,grid,stored):
        a=np.asarray(coord[:-1],float)[:,None];b=np.asarray(coord[1:],float)[:,None]
        x=np.asarray(other[:-1],float)[:,None];y=np.asarray(other[1:],float)[:,None]
        g=np.asarray(grid,float)[None,:]
        mask=(((a<=g)&(b>g))|((b<=g)&(a>g))) & np.isfinite(a)&np.isfinite(b)&np.isfinite(x)&np.isfinite(y)&(np.abs(a-b)>1e-10)
        ratio=np.divide(g-a,b-a,out=np.zeros((len(a),len(grid)),float),where=np.abs(b-a)>1e-10)
        vals=x+ratio*(y-x)
        count=mask.sum(axis=0); single=count==1
        if not single.any():return np.nan,np.nan
        resolved=np.where(mask,vals,0).sum(axis=0)
        diff=np.abs(resolved[single]-stored[single]);diff=diff[np.isfinite(diff)]
        return (float(diff.mean()),float(np.percentile(diff,95))) if len(diff) else (np.nan,np.nan)
    records=[]
    for ii,p in enumerate(paths,1):
        try:
            with np.load(p,allow_pickle=False) as z:
                lon=np.asarray(z['lon'],float);lat=np.asarray(z['lat'],float)
            good=np.isfinite(lon)&np.isfinite(lat);lon=lon[good];lat=lat[good]
            if len(lon)<3:raise RuntimeError('有效点不足3')
            c1=count_crossings(lat,lat_grid);c2=count_crossings(lon,lon_grid)
            dt=pd.Timestamp(p.stem).normalize(); loc=obs_dates.get(dt)
            if loc is not None:
                b1mean,b1p95=unique_crossing_disagreement(lat,lon,lat_grid,obs1[loc])
                b2mean,b2p95=unique_crossing_disagreement(lon,lat,lon_grid,obs2[loc])
            else:
                b1mean=b1p95=b2mean=b2p95=np.nan
            records.append({'日期':dt,'原始轨迹点数':len(lon),
                '第一分支唯一交点偏差平均值_度':b1mean,'第一分支唯一交点偏差P95_度':b1p95,
                '第二分支唯一交点偏差平均值_度':b2mean,'第二分支唯一交点偏差P95_度':b2p95,
                '第一分支无交点比例':float(np.mean(c1==0)),'第一分支多交点比例':float(np.mean(c1>=2)),
                '第二分支无交点比例':float(np.mean(c2==0)),'第二分支多交点比例':float(np.mean(c2>=2)),
                '第二分支最大交点数':int(c2.max()),'第一分支最大交点数':int(c1.max()),'状态':'OK'})
        except Exception as exc:
            records.append({'日期':pd.Timestamp(p.stem),'状态':'ERROR','异常':repr(exc)})
        if ii%100==0:
            pd.DataFrame(records).to_csv(DATA/'04_逐日双分支经纬线交点审计_中间保存.csv',index=False,encoding='utf-8-sig')
    track_diagnostics=pd.DataFrame(records);track_diagnostics.to_csv(DATA/'04_逐日双分支经纬线交点审计.csv',index=False,encoding='utf-8-sig')
    track_diagnostics['年份']=track_diagnostics['日期'].dt.year
    trsum=track_diagnostics[track_diagnostics['状态']=='OK'].groupby('年份',as_index=False)[[
        '第一分支无交点比例','第一分支多交点比例','第二分支无交点比例','第二分支多交点比例']].mean()
    trsum.to_csv(DATA/'04_逐年双分支经纬线交点审计.csv',index=False,encoding='utf-8-sig')
    display(trsum)


In [ ]:
# ===== 4. 预先声明的可视化：中英双份、原稿风格、缺CJK字体不输出乱码 =====
CJK=None
for path in ['/mnt/c/Windows/Fonts/msyh.ttc','/mnt/c/Windows/Fonts/simhei.ttf','/mnt/c/Windows/Fonts/simsun.ttc',
             '/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc']:
    if Path(path).exists():
        try:
            font_manager.fontManager.addfont(path)
            CJK=FontProperties(fname=path).get_name();break
        except Exception:pass
if CJK is None:raise RuntimeError('未检测到可用中文字体，请先安装 sudo apt-get install fonts-noto-cjk；禁止绘制中文乱码图。')

def style(lang):
    plt.rcdefaults()
    matplotlib.rcParams.update({'font.family':[CJK,'DejaVu Sans'] if lang=='zh' else ['DejaVu Serif'],
        'axes.unicode_minus':False,'pdf.fonttype':42,'figure.facecolor':'white','axes.facecolor':'white',
        'xtick.direction':'in','ytick.direction':'in','xtick.top':True,'ytick.right':True,
        'grid.color':'#DDDDDD','grid.linestyle':':','axes.grid':True,'legend.frameon':False})

def save_bilingual(stem,func):
    for lang,folder in [('zh',FZH),('en',FEN)]:
        style(lang); fig=func(lang)
        fig.savefig(folder/f'{stem}_{"中文" if lang=="zh" else "English"}.png',dpi=cfg.FIG_DPI,bbox_inches='tight')
        fig.savefig(folder/f'{stem}_{"中文" if lang=="zh" else "English"}.pdf',bbox_inches='tight')
        plt.close(fig)

def fig_qc(lang):
    fig,ax=plt.subplots(figsize=(10,4.7))
    ax.bar(annual_flags['年份'],100*annual_flags['标记比例'],color='#6C92AB',width=.75)
    ax.set_xlabel('年份' if lang=='zh' else 'Year')
    ax.set_ylabel('宽松物理范围标记比例（%）' if lang=='zh' else 'Broad-range flag rate (%)')
    ax.set_title('逐年路径异常标记分布（不等于识别错误）' if lang=='zh' else 'Annual physical-range flags (not confirmed errors)')
    fig.tight_layout();return fig
save_bilingual('图1_年度路径物理范围标记',fig_qc)

lmod=['Matched-LSTM','R53-CapacityMatchedLSTM','EOF-TCN-LSTM','R53-ChannelConv-PCConcat']
colors=['#9AB5C8','#79A09D','#4F7594','#C88463']
mean=qa.groupby(['模型','提前期_天'],as_index=False)['横向路径误差MSE_km2'].mean();mean['RMSE_km']=np.sqrt(mean['横向路径误差MSE_km2'])
def fig_models(lang):
    fig,ax=plt.subplots(figsize=(8.5,5.2))
    labels={'Matched-LSTM':'同参数 LSTM','R53-CapacityMatchedLSTM':'扩容匹配 LSTM','EOF-TCN-LSTM':'原 EOF-TCN-LSTM','R53-ChannelConv-PCConcat':'双通道卷积+EOF旁路'}
    for m,col in zip(lmod,colors):
        q=mean[mean['模型']==m].sort_values('提前期_天')
        ax.plot(q['提前期_天'],q['RMSE_km'],'-o',color=col,markerfacecolor='white',label=labels[m] if lang=='zh' else m)
    ax.set_xticks([30,60,90]);ax.set_xlabel('预测提前期（天）' if lang=='zh' else 'Forecast lead (days)')
    ax.set_ylabel('路径 RMSE（km）' if lang=='zh' else 'Axis-path RMSE (km)')
    ax.set_title('R5.3 机制资格结果（2008与2016年）' if lang=='zh' else 'R5.3 exploratory QA (2008 and 2016)')
    ax.legend(fontsize=8);fig.tight_layout();return fig
save_bilingual('图2_R53研发资格误差',fig_models)

# 图三明确分层：不做“异常造成误差”的因果推断
sm=strata_avg[strata_avg['模型'].isin(['Matched-LSTM','R53-ChannelConv-PCConcat'])].groupby(['模型','异常目标日期'],as_index=False)['RMSE均值_km'].mean()
def fig_stratified(lang):
    fig,ax=plt.subplots(figsize=(7,4.8))
    x=np.arange(2);width=.32
    for j,(m,col) in enumerate([('Matched-LSTM','#79A3B9'),('R53-ChannelConv-PCConcat','#C88463')]):
        vals=[float(sm[(sm['模型']==m)&(sm['异常目标日期']==b)]['RMSE均值_km'].iloc[0]) for b in [False,True]]
        ax.bar(x+(j-.5)*width,vals,width,color=col,label=('同参数LSTM' if j==0 else '双通道+EOF旁路') if lang=='zh' else m)
    ax.set_xticks(x);ax.set_xticklabels(['未标记','宽松范围异常'] if lang=='zh' else ['Unflagged','Flagged'])
    ax.set_ylabel('各年份/提前期/种子平均 RMSE（km）' if lang=='zh' else 'Stratum-mean RMSE (km)')
    ax.set_title('物理范围标记与预测误差关联（仅描述性）' if lang=='zh' else 'Association of range flags and forecast errors')
    ax.legend(fontsize=8);fig.tight_layout();return fig
save_bilingual('图3_路径异常状态与预测误差',fig_stratified)

if len(track_diagnostics) and '第二分支多交点比例' in track_diagnostics:
    g=track_diagnostics[track_diagnostics['状态']=='OK'].copy();g['年份']=g['日期'].dt.year
    gr=g.groupby('年份',as_index=False)[['第二分支无交点比例','第二分支多交点比例']].mean()
    def fig_cross(lang):
        fig,ax=plt.subplots(figsize=(10,4.6))
        ax.plot(gr['年份'],100*gr['第二分支无交点比例'],'-o',color='#4F7594',ms=3,label='无交点' if lang=='zh' else 'No crossing')
        ax.plot(gr['年份'],100*gr['第二分支多交点比例'],'-s',color='#C88463',ms=3,label='多个交点' if lang=='zh' else 'Multiple crossings')
        ax.set_xlabel('年份' if lang=='zh' else 'Year');ax.set_ylabel('第二分支固定经线比例（%）' if lang=='zh' else 'Branch-2 longitude-grid rate (%)')
        ax.set_title('第二分支单值表示的潜在失真来源' if lang=='zh' else 'Potential nonuniqueness of branch-2 longitude representation')
        ax.legend();fig.tight_layout();return fig
    save_bilingual('图4_第二分支多值与外推风险',fig_cross)
print('中文图表：',FZH,'英文图表：',FEN)


In [ ]:
# ===== 5. 诊断汇总 + 自动归档（即使发现未达标也归档） =====
import zipfile, hashlib
from IPython.display import display, FileLink, Markdown

diagnostic={
    'experiment':'R5.4 data representation audit, no retraining',
    'source':'R5.3 exploratory QA (2008,2016) and frozen axis tracks',
    'R53_candidate_qualified':False,
    'frozen_dataset_present':bool(DS.exists()),
    'track_scan_n':int(len(track_diagnostics)),
    'flagged_days':int(len(raw_flags)),
    'R53_mean_gain_PCConcat_vs_stronger_matched_LSTM_km':-0.3024550782574309,
    'claims':'Flag status is not a verified misdetection. No changes to original dataset or test labels.',
    'next_decision':'Review physical branch representation and crossing multiplicity first; no architectural escalation before attribution.',
}
(REPORT/'R54_诊断交接.json').write_text(json.dumps(diagnostic,ensure_ascii=False,indent=2),encoding='utf-8')
lines=['# R5.4 研究诊断摘要','', 'R5.3 两个候选全部未获得资格；当前不建议继续堆叠TCN模块。',
    '被宽松物理范围标记的目标日期与较高预测误差相关，但这不是错误标注的证据。',
    f'原始路径扫描条数：{len(track_diagnostics)}（0表示尚未扫描，不代表不存在）。',
    '下一步须先区分：真实多值大蛇行、坐标外推、插值失真和错误路径识别，再决定是否重建路径表示。',
    '注意：2008/2016 QA多次被实验开发接触，不能当独立盲测。']
(REPORT/'R54_研究诊断摘要.md').write_text('\n\n'.join(lines),encoding='utf-8')

archive=ROOT/'R54_上传给ChatGPT_路径表示与误差归因审计.zip'; digestpath=ROOT/(archive.name+'.sha256')
files=sorted([p for p in OUT.rglob('*') if p.is_file() and p.name != 'R54_文件清单_SHA256.csv' and p.stat().st_size<=cfg.MAX_ARCHIVE_FILE_MB*1024**2])
manifest=[]
for p in files:
    manifest.append({'文件':str(p.relative_to(OUT)),'字节':p.stat().st_size,'SHA256':hashlib.sha256(p.read_bytes()).hexdigest()})
pd.DataFrame(manifest).to_csv(REPORT/'R54_文件清单_SHA256.csv',index=False,encoding='utf-8-sig')
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED,compresslevel=5,allowZip64=True) as z:
    for p in files: z.write(p,arcname='R54_实验结果/'+p.relative_to(OUT).as_posix())
    z.write(REPORT/'R54_文件清单_SHA256.csv',arcname='R54_实验结果/报告/R54_文件清单_SHA256.csv')
    check=z.testzip()
    if check:raise RuntimeError('ZIP校验失败: '+str(check))
digest=hashlib.sha256(archive.read_bytes()).hexdigest()
digestpath.write_text(digest+'  '+archive.name+'\n',encoding='utf-8')
print('完成：',archive,'size(MB)=',archive.stat().st_size/1024**2,'\nSHA256=',digest)
display(Markdown('**实验归档已生成（本机文件）：**'))
display(FileLink(str(archive)))
print('若VS Code Webview无法打开链接，可直接在Windows资源管理器访问 D:\\KuroshioDaily1993_2025\\'+archive.name)
